# Aula 22: Árvores Aleatórias Exploradas Rapidamente (RRT) — Planejamento de Rotas em Espaço Contínuo para o Visão-AGV

## 1. Visão Geral da Aula

Nesta aula, implementaremos o algoritmo **RRT (Rapidly-exploring Random Tree)** para o planejamento de caminhos do robô **Visão-AGV** em ambientes industriais com obstáculos dinâmicos e contínuos.

### Contexto no Projeto Visão-AGV
Enquanto redes de grafos discretos (como Dijkstra/A*) são ideais para navegar em corredores predefinidos, o algoritmo **RRT** permite que o AGV realize desvios autônomos em áreas abertas (como zonas de manobra e pátios de carga) ao detectar obstáculos imprevistos (ex: paletes fora do lugar, empilhadeiras em trânsito).

**Princípios do RRT:**
1. **Amostragem Aleatória ($q_{rand}$):** Sorteia um ponto no espaço de estados continuo ($2D$).
2. **Busca do Vizinho mais Próximo ($q_{near}$):** Localiza o nó existente na árvore mais próximo de $q_{rand}$.
3. **Extensão Controlada ($q_{new}$):** Avanca uma distância limite $\Delta q$ em direção a $q_{rand}$.
4. **Checagem de Colisão:** Cancela o passo se a aresta interceptar a **Zona de Exclusão (Keep-out Zone)** de um obstáculo.


In [ ]:
# Setup Inicial e Representação Numérica dos Nós e Obstáculos do AGV

import math
import random

class NoRRT:
    """
    Representa uma posição contínua (x, y) alcançada pela árvore de navegação do AGV.
    """
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.pai = None  # Referência ao nó pai para reconstrução do caminho

class ObstaculoCircular:
    """
    Representa uma zona de exclusão/obstáculo no galpão de manufatura.
    """
    def __init__(self, x, y, raio):
        self.x = x
        self.y = y
        self.raio = raio

    def colide(self, x, y):
        """Verifica se um ponto está dentro do raio de colisão."""
        dist = math.hypot(x - self.x, y - self.y)
        return dist <= self.raio

print("Classes base de RRT inicializadas com sucesso!")

## 2. Algoritmo de Planejamento RRT (Exploração e Expansão)

Construímos a classe `PlanejadorRRT` responsável pela amostragem do espaço industrial e criação da árvore.

In [ ]:
class PlanejadorRRT:
    def __init__(self, inicio, objetivo, obstaculos, limites_mapa, passo_extensao=2.0, max_iter=1000):
        self.inicio = NoRRT(inicio[0], inicio[1])
        self.objetivo = NoRRT(objetivo[0], objetivo[1])
        self.obstaculos = obstaculos
        self.limite_x = limites_mapa[0]
        self.limite_y = limites_mapa[1]
        self.passo = passo_extensao
        self.max_iter = max_iter
        self.arvore = [self.inicio]

    def planejar(self):
        """Executa a expansão estocástica da árvore RRT."""
        for _ in range(self.max_iter):
            # 1. Amostragem (Bias de 10% em direção ao objetivo final)
            if random.random() < 0.10:
                q_rand = (self.objetivo.x, self.objetivo.y)
            else:
                q_rand = (random.uniform(0, self.limite_x), random.uniform(0, self.limite_y))

            # 2. Encontra nó mais próximo na árvore existente
            q_near = min(self.arvore, key=lambda no: math.hypot(no.x - q_rand[0], no.y - q_rand[1]))

            # 3. Calcula novo ponto (q_new) estendendo na direção de q_rand
            angulo = math.atan2(q_rand[1] - q_near.y, q_rand[0] - q_near.x)
            x_new = q_near.x + self.passo * math.cos(angulo)
            y_new = q_near.y + self.passo * math.sin(angulo)

            # 4. Checagem de colisões com os obstáculos do galpão
            if not any(obs.colide(x_new, y_new) for obs in self.obstaculos) and not self._linha_colide(q_near.x, q_near.y, x_new, y_new):
                q_new = NoRRT(x_new, y_new)
                q_new.pai = q_near
                self.arvore.append(q_new)

                # Checa se alcançou a zona de entrega/objetivo
                if math.hypot(q_new.x - self.objetivo.x, q_new.y - self.objetivo.y) <= self.passo:
                    self.objetivo.pai = q_new
                    self.arvore.append(self.objetivo)
                    return self._reconstruir_caminho()

        return None

    def _linha_colide(self, x1, y1, x2, y2):
        """Verifica colisão ao longo do segmento do trecho."""
        passos = 10
        for i in range(passos + 1):
            t = i / passos
            x = x1 + t * (x2 - x1)
            y = y1 + t * (y2 - y1)
            if any(obs.colide(x, y) for obs in self.obstaculos):
                return True
        return False

    def _reconstruir_caminho(self):
        caminho = []
        atual = self.objetivo
        while atual is not None:
            caminho.append((atual.x, atual.y))
            atual = atual.pai
        return caminho[::-1]

print("PlanejadorRRT implementado com sucesso!")

## 3. Simulação de Desvios de Obstáculos em Pátio de Manobra

In [ ]:
# Execução da Simulação do AGV desviando de paletes e barreiras
random.seed(42) # Semente para reprodutibilidade determinística

pos_inicio = (2.0, 2.0)       # Estação de Recarga ST-01
pos_destino = (45.0, 45.0)    # Doca de Embarque DOC-101
limites = (50.0, 50.0)        # Dimensões da área (50m x 50m)

# Definindo obstáculos industriais (x, y, raio)
zonas_bloqueadas = [
    ObstaculoCircular(15.0, 15.0, 6.0), # Paletes de Matéria-Prima
    ObstaculoCircular(30.0, 20.0, 7.0), # Célula Robotizada em Manutenção
    ObstaculoCircular(25.0, 35.0, 5.0)  # Pilar Estrutural
]

rrt = PlanejadorRRT(pos_inicio, pos_destino, zonas_bloqueadas, limites, passo_extensao=2.5, max_iter=2000)
caminho_gerado = rrt.planejar()

print("=== SIMULAÇÃO DE NAVEGAÇÃO RRT PARA VISÃO-AGV ===")
if caminho_gerado:
    print(f"✓ Caminho autônomo gerado com {len(caminho_gerado)} waypoints contínuos!")
    print("Primeiros 5 waypoints:", [(round(x, 2), round(y, 2)) for x, y in caminho_gerado[:5]])
else:
    print("❌ Falha em encontrar caminho dentro do limite de iterações.")

## 4. Conclusão e Aplicação Prática

1. **Suporte a Ambientes Não-Estruturados:** Diferente do A* tradicional, o RRT opera em coordenadas flutuantes reais $(X, Y)$, não exigindo uma grade quadriculada pré-definida.
2. **Validação para Robôs Móveis:** Permite que o **Visão-AGV** encontre caminhos válidos desviando de obstruções dinâmicas em tempo de execução.